In [ ]:
!wget -O data.zip https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip
!unzip -o data.zip
!pip install xlrd openpyxl xgboost

--2026-11-11 16:52:32--  https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘data.zip’

data.zip                [     <=>            ]   5.28M  4.35MB/s    in 1.2s    

2026-11-11 16:52:34 (4.35 MB/s) - ‘data.zip’ saved [5539494]

Archive:  data.zip
 extracting: default of credit card clients.xls  


In [ ]:
!pip install xlrd openpyxl

In [ ]:
"""ES 678: calibration and failure-mode audit on UCI Default of Credit Card Clients.
Usage: python run_experiments.py path/to/default_of_credit_card_clients.(xls|xlsx|csv) [n_seeds]
Outputs CSVs + reliability.png in ./results
"""
import sys, os, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)

# ---------- data ----------
def load(path):
    if path.endswith(".csv"):
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path, header=1)
    df = df.rename(columns={"default payment next month": "default", "PAY_0": "PAY_1"})
    df = df.drop(columns=[c for c in ["ID"] if c in df.columns])
    df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})   # undocumented -> other
    df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})                 # undocumented -> other
    return df

CAT = ["SEX", "EDUCATION", "MARRIAGE"]

def prep(num_cols):
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
                              ("num", StandardScaler(), num_cols)])

# ---------- metrics ----------
def ece(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    return sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean())
               for b in range(bins) if (idx == b).any())

def metrics(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return dict(PR_AUC=average_precision_score(y, p), ROC_AUC=roc_auc_score(y, p),
                Brier=brier_score_loss(y, p), LogLoss=log_loss(y, p), ECE=ece(y, p))

def bootstrap_ci(y, p, n=1000, seed=0):
    rng = np.random.default_rng(seed); rows = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max(): continue
        rows.append(metrics(y[i], p[i]))
    r = pd.DataFrame(rows)
    return r.quantile([0.025, 0.975]).T.rename(columns={0.025: "lo", 0.975: "hi"})

# ---------- calibrators (fit on validation set only) ----------
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

def fit_platt(p, y):
    m = LogisticRegression(C=1e6).fit(logit(p).reshape(-1, 1), y)
    return lambda q: m.predict_proba(logit(q).reshape(-1, 1))[:, 1]

def fit_iso(p, y):
    m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y)
    return lambda q: m.predict(q)

# ---------- models ----------
def make_model(name, num_cols, cw, pos_w, seed):
    pre = prep(num_cols)
    if name == "LR":
        clf = LogisticRegression(penalty="l2", max_iter=1000, class_weight=cw)
        grid = {"clf__C": [0.01, 0.1, 1, 10]}
    elif name == "RF":
        clf = RandomForestClassifier(n_estimators=100, class_weight=cw, n_jobs=-1, random_state=seed)
        grid = {"clf__max_depth": [6, 10, 16, None], "clf__min_samples_leaf": [1, 5, 20]}
    else:
        clf = XGBClassifier(n_estimators=100, eval_metric="logloss", n_jobs=-1, random_state=seed,
                            scale_pos_weight=pos_w if cw else 1.0, tree_method="hist", device = "cuda")
        grid = {"clf__max_depth": [2, 3, 4, 6], "clf__learning_rate": [0.03, 0.1],
                "clf__subsample": [0.8, 1.0], "clf__min_child_weight": [1, 5, 20]}
    return Pipeline([("pre", pre), ("clf", clf)]), grid

def fit_tuned(name, Xtr, ytr, num_cols, cw, seed, n_iter=4):
    pos_w = (ytr == 0).sum() / (ytr == 1).sum()
    pipe, grid = make_model(name, num_cols, cw, pos_w, seed)
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)          # CV on train only
    s = RandomizedSearchCV(pipe, grid, n_iter=min(n_iter, int(np.prod([len(v) for v in grid.values()]))),
                           scoring="average_precision", cv=cv, random_state=seed, n_jobs=1)
    return s.fit(Xtr, ytr).best_estimator_

def split(df, seed):
    X, y = df.drop(columns="default"), df["default"].values
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=seed)
    Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=seed)
    return Xtr, ytr, Xva, yva, Xte, yte

In [ ]:
# ---------- main experiment ----------
def main(path, n_seeds=3):
    df = load(path); num_cols = [c for c in df.columns if c not in CAT + ["default"]]
    main_rows, probs0 = [], {}
    for seed in range(n_seeds):
        Xtr, ytr, Xva, yva, Xte, yte = split(df, seed)
        for name in ["LR", "RF", "XGB"]:
            m = fit_tuned(name, Xtr, ytr, num_cols, None, seed)
            pva, pte = m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
            variants = {"raw": pte, "platt": fit_platt(pva, yva)(pte), "isotonic": fit_iso(pva, yva)(pte)}
            for v, p in variants.items():
                main_rows.append(dict(seed=seed, model=name, calib=v, **metrics(yte, p)))
                if seed == 0: probs0[(name, v)] = p
    main = pd.DataFrame(main_rows)
    main.to_csv("results/main_all_seeds.csv", index=False)
    main.groupby(["model", "calib"]).agg(["mean", "std"]).round(4).to_csv("results/main_summary.csv")

    # bootstrap CIs (seed 0 test set)
    _, _, _, _, Xte, yte = split(df, 0)
    ci = {f"{k[0]}|{k[1]}": bootstrap_ci(yte, p) for k, p in probs0.items()}
    pd.concat(ci, names=["model|calib", "metric"]).to_csv("results/bootstrap_ci_seed0.csv")

    # reliability diagrams
    fig, ax = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
    for a, name in zip(ax, ["LR", "RF", "XGB"]):
        for v, mk in [("raw", "o-"), ("platt", "s--"), ("isotonic", "^:")]:
            p = probs0[(name, v)]; e = np.linspace(0, 1, 11)
            idx = np.clip(np.digitize(p, e[1:-1]), 0, 9)
            xs = [p[idx == b].mean() for b in range(10) if (idx == b).sum() >= 20]
            ys = [yte[idx == b].mean() for b in range(10) if (idx == b).sum() >= 20]
            a.plot(xs, ys, mk, label=v, ms=3)
        a.plot([0, 1], [0, 1], "k-", lw=.5); a.set_title(name); a.set_xlabel("predicted")
    ax[0].set_ylabel("observed default rate"); ax[0].legend(fontsize=7)
    plt.tight_layout(); plt.savefig("results/reliability.png", dpi=200)

    # sensitivity: train size x class weighting (seed 0)
    Xtr, ytr, Xva, yva, Xte, yte = split(df, 0); rows = []
    #for frac in [0.1, 0.25, 0.5, 1.0]:
    for frac in [0.25, 1.0]:
        n = int(frac * len(Xtr)); idx = np.random.default_rng(0).permutation(len(Xtr))[:n]
        Xs, ys = Xtr.iloc[idx], ytr[idx]
        for name in ["LR", "RF", "XGB"]:
            for cw in [None, "balanced"]:
                m = fit_tuned(name, Xs, ys, num_cols, cw, 0, n_iter=2)
                pva, pte = m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
                for v, p in [("raw", pte), ("isotonic", fit_iso(pva, yva)(pte))]:
                    rows.append(dict(frac=frac, model=name, class_weight=cw or "none", calib=v, **metrics(yte, p)))
    pd.DataFrame(rows).to_csv("results/sensitivity.csv", index=False)

    # subgroup audit + cost analysis (XGB, seed 0)
    pva_m = fit_tuned("XGB", Xtr, ytr, num_cols, None, 0)
    pva, pte = pva_m.predict_proba(Xva)[:, 1], pva_m.predict_proba(Xte)[:, 1]
    cal = fit_iso(pva, yva)(pte)
    t_cost = 1 / (1 + 5)                                            # c_FN=5, c_FP=1 -> t*=c_FP/(c_FP+c_FN)
    grp = pd.DataFrame({"y": yte, "raw": pte, "cal": cal,
                        "limit_band": pd.qcut(Xte["LIMIT_BAL"].values, 4, labels=["Q1", "Q2", "Q3", "Q4"]),
                        "age_band": pd.cut(Xte["AGE"].values, [0, 29, 39, 49, 120], labels=["<30", "30-39", "40-49", "50+"])})
    out = []
    for g in ["limit_band", "age_band"]:
        for k, d in grp.groupby(g):
            yy = d["y"].values
            for v in ["raw", "cal"]:
                flag = d[v].values >= t_cost
                out.append(dict(grouping=g, group=k, probs=v, n=len(d), base_rate=yy.mean(), ECE=ece(yy, d[v].values),
                                FNR=((~flag) & (yy == 1)).sum() / max((yy == 1).sum(), 1)))
    pd.DataFrame(out).to_csv("results/subgroups.csv", index=False)

    cost_rows = []
    for v in ["raw", "cal"]:
        flag = grp[v].values >= t_cost; yy = grp["y"].values
        fn, fp = ((~flag) & (yy == 1)).sum(), (flag & (yy == 0)).sum()
        cost_rows.append(dict(probs=v, threshold=t_cost, flagged=flag.sum(), FN=fn, FP=fp,
                              expected_cost=(5 * fn + fp) / len(yy),
                              expected_cost_from_probs=np.mean(np.where(flag, (1 - grp[v]) * 1, grp[v] * 5))))
    pd.DataFrame(cost_rows).to_csv("results/cost.csv", index=False)

    # high-confidence false negatives: lowest predicted risk among true defaulters
    fn = Xte.assign(p_cal=cal, y=yte)[yte == 1].sort_values("p_cal").head(50)
    fn.describe().T.to_csv("results/high_conf_FN_profile.csv")
    print(pd.read_csv("results/main_summary.csv"))

if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main(sys.argv[1], int(sys.argv[2]) if len(sys.argv) > 2 else 3)

In [ ]:
main("/content/default of credit card clients.xls", n_seeds=2)

   Unnamed: 0 Unnamed: 1  seed  seed.1  PR_AUC PR_AUC.1 ROC_AUC ROC_AUC.1  \
0         NaN        NaN  mean     std    mean      std    mean       std   
1       model      calib   NaN     NaN     NaN      NaN     NaN       NaN   
2          LR   isotonic   0.5  0.7071  0.4814   0.0178  0.7159    0.0087   
3          LR      platt   0.5  0.7071   0.502   0.0095   0.715    0.0063   
4          LR        raw   0.5  0.7071   0.502   0.0095   0.715    0.0063   
5          RF   isotonic   0.5  0.7071  0.5385   0.0131  0.7727    0.0147   
6          RF      platt   0.5  0.7071  0.5579    0.014  0.7743    0.0145   
7          RF        raw   0.5  0.7071  0.5579    0.014  0.7743    0.0145   
8         XGB   isotonic   0.5  0.7071  0.5361   0.0189  0.7726    0.0174   
9         XGB      platt   0.5  0.7071  0.5524    0.019  0.7742    0.0167   
10        XGB        raw   0.5  0.7071  0.5524    0.019  0.7742    0.0167   

     Brier Brier.1 LogLoss LogLoss.1     ECE   ECE.1  
0     mean     std  

In [ ]:
df = load("default of credit card clients.xls")   # note: load() already merged codes
raw = pd.read_excel("default of credit card clients.xls", header=1)
print("n rows:", len(raw), "| default rate:", raw["default payment next month"].mean().round(4))
print("education 0/5/6:", raw["EDUCATION"].isin([0,5,6]).sum())
print("marriage 0:", (raw["MARRIAGE"]==0).sum())
pay = raw[["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]]
print("client-months with PAY=-2:", (pay==-2).sum().sum(), "| PAY=0:", (pay==0).sum().sum())

n rows: 30000 | default rate: 0.2212
education 0/5/6: 345
marriage 0: 54
client-months with PAY=-2: 24415 | PAY=0: 95919
